# Week 8 — Evaluation Expansion

Evaluate the June 2026 held-out predictions overall and by price band.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

PREDICTION_FILES = {
    'XGBoost': Path('../week7/outputs/xgboost_test_predictions.csv'),
    'LightGBM': Path('../week7/outputs/lightgbm_test_predictions.csv'),
}

def evaluate(actual, predicted):
    actual, predicted = np.asarray(actual, float), np.asarray(predicted, float)
    ape = np.abs(predicted - actual) / np.abs(actual)
    return {'R2': r2_score(actual, predicted), 'RMSE': np.sqrt(mean_squared_error(actual, predicted)), 'MAE': mean_absolute_error(actual, predicted), 'MAPE': ape.mean() * 100, 'MdAPE': np.median(ape) * 100}


In [2]:
overall, bands = [], []
for model, path in PREDICTION_FILES.items():
    if not path.exists():
        print(f'Skipping {model}: {path} is not available')
        continue
    df = pd.read_csv(path)[['Actual', 'Predicted']].dropna()
    overall.append({'Model': model, 'Number of Homes': len(df), **evaluate(df.Actual, df.Predicted)})
    df['Price Band'] = pd.qcut(df.Actual, 5, labels=['Lowest 20%', '20%–40%', '40%–60%', '60%–80%', 'Highest 20%'])
    for band, group in df.groupby('Price Band', observed=True):
        bands.append({'Model': model, 'Price Band': str(band), 'Number of Homes': len(group), 'Median Actual Price': group.Actual.median(), **evaluate(group.Actual, group.Predicted)})

if not overall:
    raise FileNotFoundError('Run Week 7 before this notebook.')
metrics_summary = pd.DataFrame(overall).sort_values('R2', ascending=False)
price_band_summary = pd.DataFrame(bands)
metrics_summary.to_csv('metrics_summary.csv', index=False)
price_band_summary.to_csv('price_band_summary.csv', index=False)
display(metrics_summary)
display(price_band_summary)


,Model,Number of Homes,R2,RMSE,MAE,MAPE,MdAPE
1,LightGBM,12853,0.765004,744924.341962,236660.963167,23.859080,11.099833
0,XGBoost,12853,0.762224,749317.342061,271761.135162,28.183804,14.047474


,Model,Price Band,Number of Homes,Median Actual Price,R2,RMSE,MAE,MAPE,MdAPE
0,XGBoost,Lowest 20%,2616,449000.0,-1.954983,1.794366e+05,120126.841991,68.119664,17.860886
1,XGBoost,20%–40%,2546,699000.0,-7.697903,1.939531e+05,124807.565534,18.174956,12.300809
2,XGBoost,40%–60%,2560,925000.0,-7.903845,2.532327e+05,154187.571062,16.479016,11.335790
3,XGBoost,60%–80%,2560,1340000.0,-3.046677,3.216231e+05,227748.987688,16.738944,12.586409
4,XGBoost,Highest 20%,2571,2398000.0,0.641948,1.603147e+06,732468.455597,20.511077,17.448018
5,LightGBM,Lowest 20%,2616,449000.0,-0.685315,1.355109e+05,84198.698420,58.209499,12.182349
6,LightGBM,20%–40%,2546,699000.0,-4.252419,1.507194e+05,92962.996289,13.550355,9.293727
7,LightGBM,40%–60%,2560,925000.0,-6.988524,2.398635e+05,130840.656373,13.914583,9.663316
8,LightGBM,60%–80%,2560,1340000.0,-2.122386,2.825149e+05,196477.874769,14.450660,10.758745
9,LightGBM,Highest 20%,2571,2398000.0,0.638311,1.611269e+06,679471.153733,18.386030,14.640822


## Interpretation

Use MdAPE to compare typical proportional error across price bands. Compare RMSE with MAPE because very expensive homes can dominate RMSE.